# 02 — Model Selection & Cross-Asset Validation

**Part 1**: Grid search on SPY + threshold calibration + comparison table + feature importances  
**Part 2**: Cross-asset validation on SPY / DIA / QQQ

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_data
from src.features import build_features, COLUMN_ORDER
from src.models import RFCModel, XGBModel

---
## Part 1 — Model Search & Comparison (SPY)
### Step 1 — Data Preparation

In [ ]:

df_raw = load_data('SPY')
X_raw = build_features(df_raw)

df_spy = X_raw.copy()

ma50  = df_raw['Close'].rolling(50).mean()
ma200 = df_raw['Close'].rolling(200).mean()
gc    = (ma50 > ma200).astype(int)
gc_clean = gc.reset_index(drop=True)

transition = np.zeros(len(df_spy), dtype=int)
label = gc_clean.iloc[0]
for i in range(1, len(gc_clean)):
    if gc_clean.iloc[i] != label:
        label = gc_clean.iloc[i]
        start = max(0, i - 30)
        transition[start:i] = 1

df_spy['Transition'] = transition

df_spy = df_spy.dropna().reset_index(drop=True)


X = df_spy[COLUMN_ORDER]
y = df_spy['Transition']

split = int(len(df_spy) * 0.75)

X_train = X.iloc[:split]
X_test  = X.iloc[split:]
y_train = y.iloc[:split]
y_test  = y.iloc[split:]

print(f"Train size: {len(X_train)}  |  positives: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Test size:  {len(X_test)}   |  positives: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
X_train.head(500)

Train size: 4563  |  positives: 528 (11.6%)
Test size:  1521   |  positives: 150 (9.9%)


,Return,Volatility,Cumulated_Return_5d,RSI14,Volume_ROC,ATR,VIX_spike,Distance_GC,MA_velocity,MA50_slope,Distance_normalized,MA_cross_momentum
0,0.004467,0.016713,0.016960,46.488303,-43.405512,2.234630,1.204567,0.008671,-0.559688,-0.011881,0.897106,56.401998
1,-0.006671,0.016545,0.035946,42.632071,-6.475955,2.254461,1.181164,0.007573,-0.525659,-0.012497,0.773328,78.072450
2,-0.023506,0.017220,0.015363,37.116537,100.083207,2.342288,1.290078,0.006090,-0.496998,-0.013276,0.612883,111.845357
3,0.002752,0.016896,-0.018843,42.134979,-6.681045,2.284917,1.375873,0.004669,-0.527376,-0.013361,0.463350,136.774833
4,0.018975,0.017493,-0.004468,48.775137,116.568323,2.317498,1.265942,0.003368,-0.554406,-0.013777,0.329129,185.006780
...,...,...,...,...,...,...,...,...,...,...,...,...
495,-0.024239,0.027371,0.108195,51.334649,17.412655,2.021277,1.000913,-0.133283,0.337906,-0.009746,-6.334763,3.166481
496,0.019873,0.026973,0.094753,59.368713,6.010676,2.010570,0.950624,-0.132059,0.454879,-0.009155,-6.282260,3.994445
497,0.004192,0.026969,0.053232,61.863553,-35.087501,2.023604,0.937234,-0.131280,0.475287,-0.008949,-6.256608,4.157015
498,0.017260,0.027022,0.065461,58.405717,-31.755362,1.959828,0.926530,-0.130295,0.476080,-0.007727,-6.220210,3.756727


In [78]:
rfc = RFCModel(
    n_estimators=500,
    max_depth=6,
    min_samples_split=6,
    min_samples_leaf=3
)
rfc.fit(X_train, y_train)

y_proba_rfc = rfc.predict_proba(X_test)
print("Threshold | Precision | Recall |  F1   | N_pred")
print("-" * 52)
for t in np.arange(0.3, 0.95, 0.05):
    pred = (y_proba_rfc >= t).astype(int)
    if pred.sum() > 0:
        p = precision_score(y_test, pred, zero_division=0)
        r = recall_score(y_test, pred, zero_division=0)
        f = f1_score(y_test, pred, zero_division=0)
        print(f"  {t:.2f}    |   {p:.3f}   |  {r:.3f} | {f:.3f} | {pred.sum()}")

Threshold | Precision | Recall |  F1   | N_pred
----------------------------------------------------
  0.30    |   0.340   |  0.933 | 0.498 | 412
  0.35    |   0.357   |  0.913 | 0.513 | 384
  0.40    |   0.392   |  0.900 | 0.547 | 344
  0.45    |   0.428   |  0.847 | 0.568 | 297
  0.50    |   0.466   |  0.813 | 0.592 | 262
  0.55    |   0.535   |  0.807 | 0.644 | 226
  0.60    |   0.631   |  0.787 | 0.700 | 187
  0.65    |   0.686   |  0.727 | 0.706 | 159
  0.70    |   0.758   |  0.627 | 0.686 | 124
  0.75    |   0.815   |  0.500 | 0.620 | 92
  0.80    |   0.811   |  0.200 | 0.321 | 37
  0.85    |   0.000   |  0.000 | 0.000 | 4


In [75]:
xgb = XGBModel(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.6,
    colsample_bytree=0.7,
    eval_metric='logloss',
    verbosity=0
)
xgb.fit(X_train, y_train)

y_proba_xgb = xgb.predict_proba(X_test)
print("Threshold | Precision | Recall |  F1   | N_pred")
print("-" * 52)
for t in np.arange(0.3, 0.95, 0.05):
    pred = (y_proba_xgb >= t).astype(int)
    if pred.sum() > 0:
        p = precision_score(y_test, pred, zero_division=0)
        r = recall_score(y_test, pred, zero_division=0)
        f = f1_score(y_test, pred, zero_division=0)
        print(f"  {t:.2f}    |   {p:.3f}   |  {r:.3f} | {f:.3f} | {pred.sum()}")

Threshold | Precision | Recall |  F1   | N_pred
----------------------------------------------------
  0.30    |   0.474   |  0.907 | 0.622 | 287
  0.35    |   0.482   |  0.907 | 0.630 | 282
  0.40    |   0.487   |  0.887 | 0.629 | 273
  0.45    |   0.502   |  0.867 | 0.636 | 259
  0.50    |   0.506   |  0.853 | 0.635 | 253
  0.55    |   0.516   |  0.840 | 0.640 | 244
  0.60    |   0.528   |  0.827 | 0.644 | 235
  0.65    |   0.544   |  0.827 | 0.656 | 228
  0.70    |   0.573   |  0.813 | 0.672 | 213
  0.75    |   0.606   |  0.800 | 0.690 | 198
  0.80    |   0.650   |  0.793 | 0.715 | 183
  0.85    |   0.720   |  0.773 | 0.746 | 161
  0.90    |   0.800   |  0.720 | 0.758 | 135


In [ ]:
def compute_transition(df_raw):
    ma50  = df_raw['Close'].rolling(50).mean()
    ma200 = df_raw['Close'].rolling(200).mean()
    gc    = (ma50 > ma200).astype(int)
    transition = np.zeros(len(df_raw), dtype=int)
    label = gc.iloc[0]
    for i in range(1, len(gc)):
        if gc.iloc[i] != label:
            label = gc.iloc[i]
            start = max(0, i - 30)
            transition[start:i] = 1
    return pd.Series(transition, name='Transition')


def prepare_asset(ticker, split=0.75):
    df_raw = load_data(ticker)
    X_raw  = build_features(df_raw)
    df     = X_raw.copy()
    df['Transition'] = compute_transition(df_raw).values
    df     = df.dropna().reset_index(drop=True)
    split_idx = int(len(df) * split)
    X = df[COLUMN_ORDER]
    y = df['Transition']
    return X.iloc[split_idx:], y.iloc[split_idx:]


def validate_cross_assets(rfc_model, xgb_model, thresh_rfc, thresh_xgb, tickers):
    results = []
    for ticker in tickers:
        X_test, y_test = prepare_asset(ticker)
        print(f"\n{'='*50}")
        print(f"Asset: {ticker} | positives: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

        for model, thresh, name in [
            (rfc_model, thresh_rfc, 'RFC'),
            (xgb_model, thresh_xgb, 'XGB')
        ]:
            proba = model.predict_proba(X_test)
            pred  = (proba >= thresh).astype(int)
            results.append({
                'Asset':     ticker,
                'Model':     name,
                'Precision': round(precision_score(y_test, pred, zero_division=0), 3),
                'Recall':    round(recall_score(y_test, pred, zero_division=0), 3),
                'F1':        round(f1_score(y_test, pred, zero_division=0), 3),
                'N_pred':    int(pred.sum())
            })

    df_results = pd.DataFrame(results)
    print("\n" + "="*60)
    print("CROSS-ASSET VALIDATION SUMMARY")
    print("="*60)
    print(df_results.to_string(index=False))
    print(f"\nMean F1 RFC: {df_results[df_results['Model']=='RFC']['F1'].mean():.3f}")
    print(f"Mean F1 XGB: {df_results[df_results['Model']=='XGB']['F1'].mean():.3f}")
    return df_results


results = validate_cross_assets(rfc, xgb, thresh_rfc=0.65, thresh_xgb=0.90, tickers=['SPY', 'DIA', 'QQQ'])


Asset: SPY | positives: 150 (9.9%)

Asset: DIA | positives: 157 (10.3%)

Asset: QQQ | positives: 132 (8.7%)

CROSS-ASSET VALIDATION SUMMARY
Asset Model  Precision  Recall    F1  N_pred
  SPY   RFC      0.686   0.727 0.706     159
  SPY   XGB      0.800   0.720 0.758     135
  DIA   RFC      0.522   0.682 0.591     205
  DIA   XGB      0.734   0.669 0.700     143
  QQQ   RFC      0.892   0.750 0.815     111
  QQQ   XGB      0.906   0.727 0.807     106

Mean F1 RFC: 0.704
Mean F1 XGB: 0.755
